# Lab 1 — Query transformation + re-ranking (EP11)

From EP10 Lab 1: a long question with the book's title and the author's name pushed the "library card" chunk down to **rank 8** — so RAG couldn't answer.
BM25, hybrid search and other chunking in EP10 didn't fix it either

The EP11 theory video gave us two kinds of tools (§16.5.1–16.5.2)
- **Transform the question before searching:** HyDE · multi-query + RAG-Fusion · step-back
- **Re-order after searching:** cross-encoder re-ranking

**What you'll do**
1. Rebuild EP10's library (textbook pages 292–374, 141 chunks)
2. **HyDE:** let the model draft an answer, and search with the draft
3. **Multi-query + RAG-Fusion:** several queries merged with RRF — and the prompt that makes it miss
4. **Re-ranking:** let a cross-encoder read the top 20 chunks and re-order them
5. Measure every method on EP10's 20 test questions

Every answer from the model comes from a real run.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §16.5 — CC BY-SA 4.0

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | sign up at openrouter.ai → Keys → Create Key, then add credit (this lab costs about 7 US cents) |
| Packages | `openai`, `python-dotenv`, `pypdf`, `numpy`, `tokenizers`, `requests` (installed in the next cell) |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

⚠️ Never put the key in the code, and never share your `.env` file

In [ ]:
%pip install -q "openai>=1.50" python-dotenv "pypdf>=6" numpy "tokenizers==0.23.3" requests

## 1. Three models: one answers, one embeds, one re-ranks

**The answering model `google/gemma-4-26b-a4b-it`** — the same model as every episode, pinned to Makora.
USD 0.09 per million input tokens and USD 0.28 per million output tokens
· The `:free` version doesn't work (tested 2026-11-22: 3 out of 3 calls got 429 "rate-limited")

**The embedding model `qwen/qwen3-embedding-8b`** — the same as EP10, pinned to Nebius

**The re-ranker `qwen/qwen3-reranker-8b`** — an open cross-encoder from the same family, called through OpenRouter's `/api/v1/rerank`
endpoint with the same key, pinned to Fireworks · about USD 0.20 per million tokens (re-ranking 20 chunks ≈ 0.2 cents)
· The book uses a BGE re-ranker through `sentence-transformers`, which downloads the model to your machine — here we call an API instead, no GPU needed

Every request's real cost is added up in `COST`

In [ ]:
import os, urllib.request, requests
import numpy as np
from openai import OpenAI
from pypdf import PdfReader
from tokenizers import Tokenizer
from concurrent.futures import ThreadPoolExecutor

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

In [ ]:
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False},
         "provider": {"order": ["makora"], "allow_fallbacks": False}}
EMBED_MODEL = "qwen/qwen3-embedding-8b"
EMBED_EXTRA = {"provider": {"order": ["nebius"], "allow_fallbacks": False}}
RERANK_MODEL = "qwen/qwen3-reranker-8b"
RERANK_PROVIDER = {"order": ["fireworks"], "allow_fallbacks": False}

In [ ]:
KEY = load_key()
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=KEY, max_retries=5)
COST = {"chat": 0.0, "embed": 0.0, "rerank": 0.0}

def ask(prompt):
    r = client.chat.completions.create(model=MODEL, temperature=0, extra_body=EXTRA,
                                       messages=[{"role": "user", "content": prompt}])
    COST["chat"] += r.usage.cost
    return r.choices[0].message.content

def embed(texts):
    vectors = []
    for i in range(0, len(texts), 64):
        r = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + 64],
                                     extra_body=EMBED_EXTRA)
        COST["embed"] += r.usage.cost
        vectors += [d.embedding for d in r.data]
    V = np.array(vectors)
    return V / np.linalg.norm(V, axis=1, keepdims=True)

## 2. The same library, and the question that missed

The same library as EP10 Lab 1: textbook v1 pages 292–374 → 512-token chunks with a 64-token overlap → 141 chunks → embeddings

In [ ]:
urllib.request.urlretrieve("https://arxiv.org/pdf/2606.24937v1", "hitchhikers-guide-v1.pdf")
reader = PdfReader("hitchhikers-guide-v1.pdf")
HEADER = "H. Roitman — The Hitchhiker’s Guide to Agentic AI: From Foundations to Systems\n"
pages = [{"page": n, "text": reader.pages[n - 1].extract_text().replace(HEADER, "")}
         for n in range(292, 375)]                      # chapters 15–19
tok = Tokenizer.from_pretrained(MODEL)

def chunk_page(page, W=512, O=64):
    ids = tok.encode(page["text"], add_special_tokens=False).ids
    return [{"source": "hitchhikers-guide-v1.pdf", "page": page["page"],
             "text": tok.decode(ids[s:s + W])}
            for s in range(0, max(len(ids) - O, 1), W - O)]

chunks = [c for p in pages for c in chunk_page(p)]
E = embed([c["text"] for c in chunks])
print(len(chunks), "chunks", E.shape)

In [ ]:
def search(query_vector):
    return list(np.argsort(-(E @ query_vector)))        # every chunk, best first

def card_rank(order):
    return [r for r, i in enumerate(order, 1) if "library card" in chunks[i]["text"]][0]

question = ("In Roitman's book The Hitchhiker's Guide to Agentic AI, "
            "what analogy does the RAG chapter use to explain RAG?")
dense = search(embed([question])[0])
print("dense: rank", card_rank(dense))

## 3. HyDE: search with a hypothetical answer

Let the model **draft an answer** first, then embed the draft instead of the question (Equation 16.20).
The draft is written like a textbook, so it can land closer to the right chunk than the question does — **even when the draft is wrong**

In [ ]:
def hyde(query):
    draft = ask("Write a short passage (about 100 words) from a textbook that answers this question. "
                "Write it as the textbook would, even if you are not sure.\n\nQuestion: " + query)
    return draft, search(embed([draft])[0])

draft, by_hyde = hyde(question)
print(draft, "\n")
print("HyDE: rank", card_rank(by_hyde))

## 4. Multi-query + RAG-Fusion

Let the model write 3 new queries → search with each (plus the original) → merge the rankings with **RRF** (the same formula as EP10 Lab 2, k = 60).
Chunks ranked well in many lists rise to the top

In [ ]:
def variants(query, extra=""):
    out = ask("Write 3 different search queries that could find the answer to this question "
              f"in a textbook.{extra} One per line, no numbering, nothing else.\n\nQuestion: " + query)
    return [line.strip(" -•\t") for line in out.splitlines() if line.strip()][:3]

def rrf(lists, k=60):
    score = {}
    for order in lists:
        for rank, i in enumerate(order, 1):
            score[i] = score.get(i, 0) + 1 / (k + rank)
    return sorted(score, key=score.get, reverse=True)

def rag_fusion(query, extra=""):
    queries = variants(query, extra)
    lists = [search(embed([query])[0])] + [search(v) for v in embed(queries)]
    return queries, lists, rrf(lists)

queries, lists, fused = rag_fusion(question)
for q, order in zip(queries, lists[1:]):
    print(f"rank {card_rank(order):>3}  {q}")
print("RAG-Fusion: rank", card_rank(fused))

In [ ]:
NO_NAMES = " Leave out book titles and author names; keep only the topic."
queries, lists, fused = rag_fusion(question, NO_NAMES)
for q, order in zip(queries, lists[1:]):
    print(f"rank {card_rank(order):>3}  {q}")
print("RAG-Fusion: rank", card_rank(fused))

## 5. Re-ranking with a cross-encoder

First-stage search with embeddings (fast) → hand the **top 20 chunks** to a cross-encoder, which reads each **question–chunk pair** and scores it again (§16.5.2).
A cross-encoder has no precomputed vectors, so it can't search the whole library, but it's more accurate

In [ ]:
def rerank(query, order, n=20):
    top = order[:n]
    r = requests.post("https://openrouter.ai/api/v1/rerank",
                      headers={"Authorization": f"Bearer {KEY}"},
                      json={"model": RERANK_MODEL, "provider": RERANK_PROVIDER, "query": query,
                            "documents": [chunks[i]["text"] for i in top]}, timeout=60)
    r.raise_for_status()
    data = r.json()
    COST["rerank"] += data["usage"]["cost"]
    return [top[x["index"]] for x in data["results"]], [x["relevance_score"] for x in data["results"]]

reranked, scores = rerank(question, dense)
print("re-ranked: rank", card_rank(reranked))
for i, s in zip(reranked[:3], scores[:3]):
    print(f"{s:.3f}  page {chunks[i]['page']}  {chunks[i]['text'][:60]!r}")

In [ ]:
SYSTEM_PROMPT = """You are a helpful assistant. Answer the question using ONLY
the provided context. If the context does not contain enough information,
say so explicitly. Cite your sources using [Doc N] notation."""

def build_rag_prompt(query, chunks):
    context_str = "\n\n".join(
        f"[Doc {i+1}] (Source: {c['source']}, Page: {c.get('page', 'N/A')})\n{c['text']}"
        for i, c in enumerate(chunks))
    return f"""{SYSTEM_PROMPT}

Context:
{context_str}

Question: {query}

Answer:"""

In [ ]:
print(ask(build_rag_prompt(question, [chunks[i] for i in reranked[:3]])))

## 6. Measure on 20 questions

One question proves nothing yet — use the same 20 test questions as EP10 Lab 3 (each with its right pages).
**hit@3** = is a right page among the top three chunks?

In [ ]:
TESTS = [  # (question, the pages that answer it) — from EP10 Lab 3
    ("What is PLAID?", {301}), ("What is HyDE?", {303}), ("What does FAISS stand for?", {298}),
    ("How can I search millions of vectors quickly?", {298}),
    ("Which method compares every query token with every document token?", {300}),
    ("How do I make keyword search understand synonyms?", {299}),
    ("When should I fine-tune a model instead of looking things up?", {296}),
    ("Which chunks should I give the model if small ones retrieve better but lack context?", {302, 303}),
    ("How do I combine keyword search and embedding search?", {298}),
    ("What analogy does the textbook use to explain RAG?", {295}),
    ("How does the textbook add randomness to retry waits?", {357}),
    ("What does FIFO mean for conversation history?", {345}),
    ("Where should the unchanging part of a prompt go so caching works?", {358}),
    ("When should a person approve an agent's action?", {355}),
    ("How does Graph RAG find groups of related entities?", {305}),
    ("Which chunking method splits where the topic changes?", {302}),
    ("Which frameworks are compared in Table 18.1?", {359}),
    ("What is the difference between episodic and semantic memory?", {321, 322}),
    ("What does an evaluator-optimizer workflow do?", {370, 371}),
    ("How does the harness notice an agent repeating the same tool call?", {357, 363}),
]

In [ ]:
def hit3(order, gold):
    return any(chunks[i]["page"] in gold for i in order[:3])

def methods(case):
    q, gold = case
    d = search(embed([q])[0])
    _, h = hyde(q)
    _, _, f = rag_fusion(q, NO_NAMES)
    return {"dense": hit3(d, gold), "HyDE": hit3(h, gold), "RAG-Fusion": hit3(f, gold),
            "re-rank": hit3(rerank(q, d)[0], gold), "HyDE + re-rank": hit3(rerank(q, h)[0], gold)}

with ThreadPoolExecutor(5) as pool:
    rows = list(pool.map(methods, TESTS))
for name in rows[0]:
    print(f"{name:15} {sum(r[name] for r in rows):2} / 20")

In [ ]:
print({k: f"${v:.4f}" for k, v in COST.items()})

## Summary

| Method | The long question ("library card" chunk) | hit@3 on 20 questions |
|---|---|---|
| dense (EP10) | rank 8 | 12 |
| HyDE | rank 1 — though the draft is wrong ("open-book") | 15–16 |
| RAG-Fusion, plain prompt | rank 9 — the queries keep the book's title | — |
| RAG-Fusion, names left out | top 3 | 13–15 |
| re-rank the top 20 | rank 1 (score ≈ 1) | 18 |
| HyDE + re-rank | — | 19 |

**Exercises**
1. **Step-back:** write a prompt that makes the model ask a broader question about the underlying principle, and search with it — run it several times: is the rank stable?
2. Change `n=20` in `rerank` to 5 and to 50 — does hit@3 change, and how much does the cost change?
3. **Contextual compression:** have Gemma remove the sentences unrelated to the question from the top three chunks before answering — how many tokens shorter is the prompt?

**Next — Lab 2:** agentic RAG — let an agent decide where to search, check whether the documents are enough, and search again if they aren't